# Octopus Remote Scanner — Colab GPU Edition
Scans aquarium cameras with CLIP on GPU, downloads only octopus-containing segments.
Results are saved to Google Drive.

**Runtime: GPU (T4 or better)**

In [6]:
# ── Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/octopus_segments'  # change if needed
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output: /content/drive/MyDrive/octopus_segments


In [7]:
# ── Install dependencies ──────────────────────────────────────────
!pip install transformers -q
!apt-get install -y ffmpeg -q

import subprocess, threading, re, time, logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('scanner')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
Device: cuda
GPU: NVIDIA L4


In [8]:
# ── Config ────────────────────────────────────────────────────────
BASE_URL   = 'https://repo.octopus-intelligence.org/public'
SESSION    = 'O-vulgaris-Nity-2026-2-20--'
USERNAME   = 'octopus'
PASSWORD   = 'communication42'

ALL_CAMERAS = ['Left Top', 'Right Back', 'Right Front', 'Right Left', 'Right Right', 'Right Top']

DATES         = ['2026-02-20', '2026-02-21', '2026-02-22', '2026-02-23', '2026-02-24']
MAX_PER_DAY   = 8       # video timestamps to check per day
TARGET        = 15      # stop after this many confirmed videos
THRESHOLD     = 0.70     # CLIP octopus detection threshold
MIN_DURATION  = 5.0     # minimum segment length in seconds
SCAN_FPS      = 0.2     # frames per second to scan (1 frame / 5s)
MAJORITY      = 2       # min cameras that must detect octopus

In [9]:
# ── Load CLIP ─────────────────────────────────────────────────────
TEXT_PROMPTS = ['an octopus in an aquarium tank', 'empty aquarium tank with rocks and no animals']

t0 = time.perf_counter()
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()

text_inputs = clip_proc(text=TEXT_PROMPTS, return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    # use sub-model directly — get_text_features() return type varies across transformers versions
    text_out   = clip_model.text_model(**text_inputs)
    text_feats = clip_model.text_projection(text_out.pooler_output)
    text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

print(f'CLIP loaded in {time.perf_counter()-t0:.1f}s on {device}')
print(f'Text features shape: {text_feats.shape}')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP loaded in 5.9s on cuda
Text features shape: torch.Size([2, 512])


In [12]:
# ── Core functions ────────────────────────────────────────────────
import queue
from collections import defaultdict

def auth_url(url):
    return url.replace('https://', f'https://{USERNAME}:{PASSWORD}@')


def list_camera_urls(date):
    import urllib.parse
    session_url = f'{BASE_URL}/{SESSION}/'
    result = {}
    for cam in ALL_CAMERAS:
        cam_enc = urllib.parse.quote(cam)
        listing = f'{session_url}{cam_enc}/Local/{date}/'
        proc = subprocess.run(
            ['curl', '-s', '--user', f'{USERNAME}:{PASSWORD}', listing],
            capture_output=True, text=True
        )
        files = re.findall(r'href="([^"]+\.mp4)"', proc.stdout)
        result[cam] = [f'{listing}{f}' for f in files]
    return result


def stream_frames(url, scan_fps=SCAN_FPS, size=224):
    cmd = [
        'ffmpeg', '-loglevel', 'error',
        '-i', auth_url(url),
        '-vf', f'fps={scan_fps},scale={size}:{size}',
        '-f', 'image2pipe', '-vcodec', 'rawvideo', '-pix_fmt', 'rgb24', '-',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    frame_size = size * size * 3
    ts, interval = 0.0, 1.0 / scan_fps
    while True:
        raw = proc.stdout.read(frame_size)
        if len(raw) < frame_size:
            break
        yield ts, np.frombuffer(raw, dtype=np.uint8).reshape((size, size, 3)).copy()
        ts += interval
    proc.stdout.close(); proc.wait()


def score_batch(frames):
    imgs   = [Image.fromarray(f) for f in frames]
    inputs = clip_proc(images=imgs, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        vision_out = clip_model.vision_model(**inputs)
        feats      = clip_model.visual_projection(vision_out.pooler_output)
        feats      = feats / feats.norm(dim=-1, keepdim=True)
    logits = (feats @ text_feats.T) * clip_model.logit_scale.exp()
    return logits.softmax(dim=-1)[:, 0].cpu().tolist()


def scan_all_cameras(active_cameras, batch_size=64):
    """
    All cameras stream into a shared queue.
    CLIP runs in batches of batch_size — prints progress so you can see it working.
    """
    SENTINEL = None
    frame_q  = queue.Queue(maxsize=512)
    cam_data = defaultdict(lambda: {'ts': [], 'scores': []})
    t0       = time.perf_counter()

    def producer(cam, url):
        try:
            for ts, frame in stream_frames(url):
                frame_q.put((cam, ts, frame))
        except Exception as e:
            print(f'  [{cam}] stream error: {e}')
        finally:
            frame_q.put((cam, SENTINEL, SENTINEL))

    threads = [
        threading.Thread(target=producer, args=(cam, url), daemon=True)
        for cam, url in active_cameras.items()
    ]
    print(f'  Starting {len(threads)} camera streams...')
    for t in threads:
        t.start()

    n_cameras    = len(active_cameras)
    done_cams    = set()
    pending_cam  = []
    pending_frm  = []
    total_scored = 0

    def flush():
        nonlocal total_scored, pending_cam, pending_frm
        if not pending_frm:
            return
        scores = score_batch(pending_frm)
        for (c, t), s in zip(pending_cam, scores):
            cam_data[c]['ts'].append(t)
            cam_data[c]['scores'].append(s)
        total_scored += len(pending_frm)
        elapsed = time.perf_counter() - t0
        print(f'  scored {total_scored} frames  ({elapsed:.0f}s)  queue={frame_q.qsize()}')
        pending_cam, pending_frm = [], []

    while len(done_cams) < n_cameras:
        try:
            cam, ts, frame = frame_q.get(timeout=15)
        except queue.Empty:
            print(f'  waiting... ({len(done_cams)}/{n_cameras} cameras done, queue={frame_q.qsize()})')
            continue

        if ts is SENTINEL:
            done_cams.add(cam)
            print(f'  [{cam}] done  ({len(done_cams)}/{n_cameras} finished)')
            if len(done_cams) == n_cameras:
                flush()
            continue

        pending_cam.append((cam, ts))
        pending_frm.append(frame)

        if len(pending_frm) >= batch_size:
            flush()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - t0
    print(f'  Scan complete: {total_scored} frames in {elapsed:.0f}s ({total_scored/elapsed:.1f} frames/sec)')

    return {
        cam: (
            np.array(d['ts'],     dtype=np.float32),
            np.array(d['scores'], dtype=np.float32),
        )
        for cam, d in cam_data.items()
    }


def detect_segments(timestamps, scores, threshold=THRESHOLD, min_dur=MIN_DURATION):
    above = scores >= threshold
    segments, in_seg, start = [], False, 0.0
    for t, flag in zip(timestamps, above):
        if flag and not in_seg:  in_seg, start = True, float(t)
        elif not flag and in_seg:
            if float(t) - start >= min_dur: segments.append((start, float(t)))
            in_seg = False
    if in_seg:
        end = float(timestamps[-1]) + 1.0
        if end - start >= min_dur: segments.append((start, end))
    return segments


def download_segment(url, start, end, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'ffmpeg', '-loglevel', 'error', '-y',
        '-ss', str(start), '-to', str(end),
        '-i', auth_url(url), '-c', 'copy', str(out_path),
    ], capture_output=True)
    return out_path

print('Functions ready.')

Functions ready.


In [ ]:
# ── Main pipeline with checkpointing ─────────────────────────────
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

confirmed = 0
skipped   = 0
t_total   = time.perf_counter()

for date in DATES:
    if confirmed >= TARGET:
        break
    print(f'\n{"="*60}\nDATE: {date}')

    camera_urls = list_camera_urls(date)
    for cam, urls in camera_urls.items():
        print(f'  {cam:<15} {len(urls)} videos')

    ref_urls = next((v for v in camera_urls.values() if v), [])
    video_names = [Path(u).stem for u in ref_urls if Path(u).stem[:2] >= '09']
    video_names = video_names[:MAX_PER_DAY]
    print(f'Checking {len(video_names)} video timestamps')

    for video_name in video_names:
        if confirmed >= TARGET:
            break

        out_dir = Path(OUTPUT_DIR) / date / video_name

        # ── Resume: skip if already processed ────────────────────
        if out_dir.exists() and any(out_dir.iterdir()):
            n = len(list(out_dir.glob('*.mp4')))
            print(f'  ✔ {date}/{video_name}  ({n} files on Drive) — skipping')
            if n > 0:
                confirmed += 1
                skipped   += 1
            continue

        hhmm = video_name[:4]
        active = {
            cam: next((u for u in urls if Path(u).stem[:4] == hhmm), None)
            for cam, urls in camera_urls.items()
        }
        active = {c: u for c, u in active.items() if u}

        print(f'\n▶  {date} / {video_name}  ({len(active)} cameras)')
        t_vid = time.perf_counter()

        # ── GPU-parallel scan ─────────────────────────────────────
        cam_results = scan_all_cameras(active, batch_size=64)

        results = {}
        for cam, (ts, sc) in cam_results.items():
            segs = detect_segments(ts, sc)
            results[cam] = (active[cam], segs)
            status = f'{len(segs)} segs  mean={sc.mean():.2f}' if len(sc) else 'no frames'
            print(f'  {cam:<15}  {status}')

        t_scan = time.perf_counter() - t_vid

        # ── Majority vote ─────────────────────────────────────────
        detected = [(cam, url, segs) for cam, (url, segs) in results.items() if segs]
        cams_str = ', '.join(c for c, _, _ in detected)
        print(f'  Majority: {len(detected)}/{len(active)}  [{cams_str}]  ({t_scan:.0f}s)')

        if len(detected) < MAJORITY:
            print(f'  ✗ No octopus — marking as visited')
            out_dir.mkdir(parents=True, exist_ok=True)
            (out_dir / '_no_octopus.txt').write_text(f'{date}/{video_name}\n')
            continue

        # ── Download → Drive ──────────────────────────────────────
        dl_jobs = [
            (url, s, e, out_dir / f"{cam.replace(' ', '_')}_{s:.0f}_{e:.0f}.mp4")
            for cam, (url, segs) in results.items() if segs
            for s, e in segs
        ]

        saved = 0
        with ThreadPoolExecutor(max_workers=6) as ex:
            futs = {ex.submit(download_segment, url, s, e, p): p
                    for url, s, e, p in dl_jobs if not p.exists()}
            for fut in as_completed(futs):
                p = fut.result()
                print(f'  ✓ {p.name}  ({p.stat().st_size/1e6:.1f}MB)')
                saved += 1

        confirmed += 1
        total_s = time.perf_counter() - t_vid
        print(f'  ✔ {saved} files saved to Drive  |  Progress: {confirmed}/{TARGET}  ({total_s:.0f}s)')

elapsed_min = (time.perf_counter() - t_total) / 60
print(f'\n{"="*60}')
print(f'COMPLETE — {confirmed} confirmed ({skipped} resumed from Drive) in {elapsed_min:.1f} min')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ── Summary of saved files ────────────────────────────────────────
import os
total_files = total_mb = 0
for root, dirs, files in os.walk(OUTPUT_DIR):
    mp4s = [f for f in files if f.endswith('.mp4')]
    if mp4s:
        folder_mb = sum(os.path.getsize(os.path.join(root, f)) for f in mp4s) / 1e6
        rel = os.path.relpath(root, OUTPUT_DIR)
        print(f'{rel:<50}  {len(mp4s):3d} files  {folder_mb:.1f}MB')
        total_files += len(mp4s)
        total_mb += folder_mb
print(f'\nTOTAL: {total_files} files  {total_mb:.1f}MB')